###Supplementary macro data — Alpha Vantage

####We will fetch India CPI, WTI, and Brent data and save them into your Unity Catalog Volumes.

In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F
from datetime import datetime

# --- CONFIGURATION ---
API_KEY = "YOURC3ODNMPQM5TL20OM"  
BASE_URL = "https://www.alphavantage.co/query"
LANDING_PATH = "/Volumes/iran_israel_capstone_project/bronze/landing_zone/macro_data"

# Ensure landing directory exists
dbutils.fs.mkdirs(LANDING_PATH)

def fetch_macro_data(function, name, interval=None):
    """Fetches data from Alpha Vantage and saves to Bronze Landing Zone"""
    params = {
        "function": function,
        "apikey": API_KEY
    }
    if interval:
        params["interval"] = interval
    
    print(f"Fetching {name}...")
    response = requests.get(BASE_URL, params=params)
    data = response.json()
    
    # Extract the data list (Alpha Vantage typically returns a 'data' or 'values' key)
    # For CPI/WTI/BRENT, the key is usually "data"
    if "data" in data:
        df_pd = pd.DataFrame(data["data"])
        
        # Save as Parquet to Landing Zone
        spark_df = spark.createDataFrame(df_pd)
        
        # Add Audit Columns as required by your project doc 
        spark_df = spark_df.withColumn("ingestion_timestamp", F.current_timestamp()) \
                           .withColumn("source_name", F.lit(name)) \
                           .withColumn("source_file", F.lit(f"alpha_vantage_{function}"))
        
        target_path = f"{LANDING_PATH}/{name.lower()}"
        spark_df.write.mode("overwrite").parquet(target_path)
        print(f"Successfully saved {name} to {target_path}")
    else:
        print(f"Error fetching {name}: {data.get('Information', 'Unknown Error')}")

# Execute Ingestion for the 3 required data points [cite: 48]
fetch_macro_data("CPI", "India_CPI", interval="monthly")
fetch_macro_data("WTI", "WTI_Crude", interval="daily")
fetch_macro_data("BRENT", "Brent_Crude_Alpha", interval="daily")

###Bronze Table Creation (Unity Catalog)

####Now we move the data from the Landing Zone into the bronze schema tables.


In [0]:
%sql
-- Switch to the correct context
USE CATALOG iran_israel_capstone_project;
USE SCHEMA bronze;

-- Create India CPI Bronze Table
CREATE OR REPLACE TABLE bronze.macro_cpi_raw AS
SELECT * FROM parquet.`/Volumes/iran_israel_capstone_project/bronze/landing_zone/macro_data/india_cpi`;

-- Create WTI Crude Bronze Table
CREATE OR REPLACE TABLE bronze.macro_wti_raw AS
SELECT * FROM parquet.`/Volumes/iran_israel_capstone_project/bronze/landing_zone/macro_data/wti_crude`;

-- Create Brent (Alpha Vantage version) Bronze Table
CREATE OR REPLACE TABLE bronze.macro_brent_alpha_raw AS
SELECT * FROM parquet.`/Volumes/iran_israel_capstone_project/bronze/landing_zone/macro_data/brent_crude_alpha`;

-- Verify the ingestion
SELECT 'CPI' as type, count(*) as record_count FROM bronze.macro_cpi_raw
UNION ALL
SELECT 'WTI' as type, count(*) as record_count FROM bronze.macro_wti_raw
UNION ALL
SELECT 'Brent' as type, count(*) as record_count FROM bronze.macro_brent_alpha_raw;

--verify 
SELECT * 
FROM bronze.macro_cpi_raw
LIMIT 100;
